# Behavior Modeling API: LimSim Implementation

## Introduction

Tactics2D provides unified reimplementations of a collection of representative traffic participant behavior models to support the development, validation, and testing of Autonomous Driving Systems (ADS) with realistic and scalable traffic interactions. LimSim is one of the default behavior models integrated into Tactics2D.

Original paper: [LimSim: A Long-term Interactive Multi-scenario Traffic Simulator](https://arxiv.org/abs/2307.12534)
Original code: [pjlab-adg/LimSim](https://github.com/pjlab-adg/LimSim)

LimSim plans by prediction, decision and planning: an interaction graph groups nearby vehicles, Monte-Carlo tree search picks a joint action for each group, and a Frenet planner turns the decision into a trajectory. It is a rule-based model - no checkpoint, no learned weights - so the whole cost is search.


## Environment Setup

Please install Tactics2D (`pip install 'tactics2d[behavior]'`) or add the Tactics2D source directory to your `PYTHONPATH`. See the [Installation Guide](https://tactics2d.readthedocs.io/en/latest/installation/) for more details. LimSim needs no checkpoint and no codebook: the model is built from its configuration alone.


## Dataset Preparation

Tactics2D does not require datasets to be stored in a fixed location. You can place a dataset in any directory and provide its path when parsing it. Adjust the paths below to match your local data layout.

| Dataset | Role here | Rate | Map |
|---------|-----------|------|-----|
| **WOMD** | the scene the four behavior demos share, so their numbers can be read side by side | 10 Hz | per-scenario, from the tfrecord |
| **inD** | off-domain: a German urban junction, recorded by a different group | 25 Hz | Lanelet2 `.osm` |
| **nuPlan** | off-domain: a city-scale map, one log carrying every lane of Boston | 20 Hz | `.gpkg` |

!!! warning "The model runs on a fixed 100 ms lattice"
    Every behavior model lays a scenario out on a fixed step - LimSim's is `step_ms = 100`. A log recorded at another rate is **resampled onto that lattice automatically** by the runner (`tactics2d.behavior.rolling_utils.to_lattice`), interpolating positions and headings; fed as-is, several of its frames would land on one index.

    No other 10 Hz dataset is set up here, so every off-domain example below is also an off-rate one. That is a gap in the data on hand, not a step the demos skipped.


## Use LimSim for Behavior Generation

Both usages below go through the same public API; the notebook only provides glue code.

| Module | Key API |
|--------|---------|
| **Dataset parsers** | `parser.parse_trajectory(...)` -> `(participants, time_range)`; `parser.parse_map(...)` -> `Map` |
| **Behavior model** | `LimSimBehaviorModel(config)` -> `.predict(...)`, `.rollout(...)` |
| **Lane routes** | `tactics2d.dataset_parser.extract_all_lane_sequences(...)` -> `{agent_id: (lane_id, ...)}` |
| **Rendering** | `BEVCamera` + `MatplotlibRenderer`, driven through `tutorial_common.render_replay_animation` |


In [1]:
import inspect
import warnings

warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.WARNING)

import seaborn as sns

import tutorial_common
from tactics2d.behavior.rolling_utils import to_lattice
from tactics2d.dataset_parser import (
    LevelXParser,
    NuPlanParser,
    WOMDParser,
    extract_all_lane_sequences,
)
from tactics2d.behavior import LimSimBehaviorModel, LimSimConfig
from tactics2d.behavior.limsim.rolling import DEFAULT_WARMUP_MS
from tactics2d.map.parser import OSMParser
from tactics2d.map.element import Map
from tactics2d.map.map_config import IND_MAP_CONFIG
from tactics2d.participant.trajectory import State, Trajectory

tutorial_common.apply_notebook_style()
sns.set_palette("husl")

pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# LimSim model config (balanced_demo ≈ original paper defaults)
LIMSIM_CFG = LimSimConfig(
    horizon_steps=50,
    dt=0.1,
    mcts_iterations=200,
    terminal_depth=4,
    interaction_distance=30.0,
    max_group_size=3,
    use_frenet_refinement=False,
)

# Prediction / display constants. The perception range and view window are shared
# with the other behavior demos and live in tutorial_common.
NUM_SECONDS = 20  # how many seconds of playback
MAX_WORKERS = 8  # parallel predict threads

## Take-Over Usage

`predict()` replaces the future of **one** vehicle and leaves everybody else on the trajectory the log recorded. It is the call all four Tactics2D behavior models share, and the one the cross-model comparison is run on.

```python
plan = model.predict(participants, map_, frame, agent_ids=[ego_id], route_map=route)
```

`frame` is the newest frame the model conditions on, in milliseconds, and the return value is `{agent_id: Trajectory}`.

!!! warning "Give it a route"
    LimSim plans along lane sequences, and `route_map` is where they come from. Without one it falls back to following the first successor lane of whatever lane the vehicle is on, which mis-routes at intersections and dead-ends. `extract_all_lane_sequences(participants, map_, frame, agent_ids=[ego_id])` builds it from the vehicle's recorded trajectory.


### Step 1: Select the Vehicle

`tutorial_common.select_ego` picks a vehicle present before the warm-up frame that survives into the second half of the scenario, but passing an explicit `ego_id` is just as valid. This demo uses the shared comparison vehicle - the same one the SMART and InterSim demos take over - so the numbers below can be read side by side.

### Step 2: Predict, and Score Against the Recorded Future

The prediction is scored against the vehicle's recorded future with `tutorial_common.displacement_errors`, once over the shared 2 s horizon and once over LimSim's own 5 s.

### Step 3: Render with BEVCamera and MatplotlibRenderer

LimSim stamps its predicted frames on the scenario's own grid, so they can be written straight onto the vehicle's trajectory. The purple gradient behind the vehicle is the path it has already driven; the green one is the prediction.


In [3]:
# ---- Take-over: one vehicle, one prediction ----
WOMD_ROOT = "../../../data/womd/uncompressed"

parser = WOMDParser()
file_name = tutorial_common.COMPARISON_FILE
folder = f"{WOMD_ROOT}/{tutorial_common.COMPARISON_SPLIT}"
participants, time_range = parser.parse_trajectory(
    tutorial_common.COMPARISON_SCENARIO, file=file_name, folder=folder
)
map_ = parser.parse_map(tutorial_common.COMPARISON_SCENARIO, file=file_name, folder=folder)
participants = to_lattice(participants, LIMSIM_CFG.step_ms)

ego_id = tutorial_common.COMPARISON_EGO
trigger = tutorial_common.COMPARISON_FRAME_MS
recorded = tutorial_common.recorded_future(participants[ego_id].trajectory, trigger)
route = extract_all_lane_sequences(participants, map_, trigger, agent_ids=[ego_id])

takeover_model = LimSimBehaviorModel(LIMSIM_CFG)
plan = takeover_model.predict(participants, map_, trigger, agent_ids=[ego_id], route_map=route)[
    ego_id
]

ade, fde, matched = tutorial_common.displacement_errors(
    plan, recorded, tutorial_common.COMPARISON_HORIZON_STEPS
)
own_ade, own_fde, own_matched = tutorial_common.displacement_errors(plan, recorded)
horizon_s = len(plan.frames) * LIMSIM_CFG.dt
print(
    f"ego: {ego_id}  |  predicted {len(plan.frames)} steps, "
    f"{min(plan.frames)}-{max(plan.frames)} ms"
)
print(f"  vs the recorded future:  ADE@2s {ade:.3f} m  FDE@2s {fde:.3f} m  ({matched} steps)")
print(
    f"                           ADE@{horizon_s:.0f}s {own_ade:.3f} m  FDE@{horizon_s:.0f}s "
    f"{own_fde:.3f} m  ({own_matched} steps)"
)

# Write the prediction onto the vehicle and animate it on the recorded scene.
ego = participants[ego_id]
ego.color = tutorial_common.EGO_COLOR
ego.trajectory._history_states = {
    frame: state for frame, state in ego.trajectory.history_states.items() if frame <= trigger
}
ego.trajectory._frames = sorted(ego.trajectory._history_states)
for frame in plan.frames:
    ego.trajectory.add_state(plan.get_state(frame))

playback_frames = [f for f in ego.trajectory.frames if f <= time_range[1]]
ani_takeover = tutorial_common.render_replay_animation(
    participants,
    map_,
    playback_frames,
    ego_id,
    plans={trigger: [(f, plan.get_state(f).x, plan.get_state(f).y) for f in plan.frames]},
    fps=1000.0 / LIMSIM_CFG.step_ms,
    title_prefix="LimSim take-over",
)
ani_takeover

ego: 8  |  predicted 50 steps, 1200-6100 ms
  vs the recorded future:  ADE@2s 0.694 m  FDE@2s 1.967 m  (20 steps)
                           ADE@5s 6.053 m  FDE@5s 18.184 m  (50 steps)


## Closed-Loop Usage

`rollout()` is the other mode: instead of one prediction, the vehicle is re-planned once per cycle and each plan's first state is committed back into its own trajectory, so every cycle plans from what the previous one committed. Every other participant keeps its recorded motion.

### Step 1: Select the Ego Vehicle

The replay is centred on one vehicle: it drives the loop, and it is the participant the take-over error is reported for.

### Step 2: Replay the Scenario Closed-Loop

```python
result = model.rollout(participants, map_, ego_id, frame_ms=1100, horizon_ms=20000)
```

`frame_ms` is the take-over frame and `horizon_ms` how far the replay runs. The result is a `LimSimRollingResult`: the replayed vehicle's whole `trajectory`, the `frames` to animate, the `plans` issued at each cycle, and `cycles`.

### Step 3: Render with BEVCamera and MatplotlibRenderer

The same renderer as above; the driver below also scores the first cycle's plan against the recorded future, which is the take-over number quoted in every example.


In [4]:
def run_takeover_scenario(
    parser,
    file_name=None,
    folder=None,
    map_path=None,
    map_config=None,
    map_file=None,
    map_folder=None,
    ego_id=None,
    ego_color="light-pink",
    fps=10,
    takeover_frame_ms=None,
    num_seconds=NUM_SECONDS,
    resolution=(1200, 800),
    **parse_kwargs,
):
    print("Parsing scenario ...")
    participants, time_range = parser.parse_trajectory(
        file=file_name, folder=folder, **parse_kwargs
    )
    participants = to_lattice(participants, LIMSIM_CFG.step_ms)

    map_ = None
    if hasattr(parser, "parse_map"):
        # Pass what the map parser takes; the scenario's own window is the
        # trajectory's business.
        takes = inspect.signature(parser.parse_map).parameters
        map_ = parser.parse_map(
            **{name: value for name, value in parse_kwargs.items() if name in takes},
            file=map_file if map_file is not None else file_name,
            folder=map_folder if map_folder is not None else folder,
        )
    if map_ is None and map_path is not None:
        print(f"  loading map from {map_path}")
        map_ = OSMParser(lanelet2=True).parse(file_path=map_path, configs=map_config)
    if map_ is None:
        map_ = Map("empty_map", scenario_type="demo")

    print(f"  participants: {len(participants)},  frames: {time_range}")
    if ego_id is None:
        ego_id = tutorial_common.select_ego(participants)
    participants[ego_id].color = ego_color
    print(f"  takeover target: {ego_id}  (color: {ego_color})")

    # The receding-horizon loop lives in the model package
    # (tactics2d.behavior.limsim.rolling); the driver only parses and renders.
    # The closed loop is goal-conditioned, the way the other three demos are: the
    # model is handed the lane sequence the recorded vehicle took. Extract it here,
    # before the runner truncates the trajectories at take-over.
    ego_frames = sorted(participants[ego_id].trajectory.frames)
    take_over = (
        takeover_frame_ms if takeover_frame_ms is not None else (ego_frames[0] + DEFAULT_WARMUP_MS)
    )
    take_over = next(frame for frame in ego_frames if frame >= take_over)
    route = extract_all_lane_sequences(participants, map_, take_over, agent_ids=[ego_id])

    model = LimSimBehaviorModel(LIMSIM_CFG)
    result = model.rollout(
        participants,
        map_,
        ego_id,
        frame_ms=take_over,
        horizon_ms=int(num_seconds * 1000),
        route_map=route,
    )

    # The take-over prediction is the first cycle's plan, scored against the
    # recorded future the way the other behavior demos score theirs.
    first_frame = min(result.plans)
    plan = Trajectory(id_=ego_id, fps=round(1000.0 / LIMSIM_CFG.step_ms, 3), stable_freq=True)
    for frame, x, y in result.plans[first_frame]:
        plan.add_state(State(frame=frame, x=x, y=y))
    recorded = tutorial_common.recorded_future(participants[ego_id].trajectory, first_frame)
    ade, fde, matched = tutorial_common.displacement_errors(
        plan, recorded, tutorial_common.COMPARISON_HORIZON_STEPS
    )
    own_ade, own_fde, own_matched = tutorial_common.displacement_errors(plan, recorded)
    horizon_s = len(plan.frames) * LIMSIM_CFG.dt

    frames = result.frames
    print(
        f"  MPC: {result.cycles} cycles  |  playback: {len(frames)} frames  "
        f"({frames[0]}-{frames[-1]} ms, ~{(frames[-1] - frames[0]) / 1000:.0f}s)"
    )
    print(
        f"  take-over vs the recorded future:  ADE@2s {ade:.3f} m  FDE@2s {fde:.3f} m  "
        f"({matched} steps)"
    )
    print(
        f"                                    ADE@{horizon_s:.0f}s {own_ade:.3f} m  "
        f"FDE@{horizon_s:.0f}s {own_fde:.3f} m  ({own_matched} steps)"
    )

    participants[ego_id].trajectory = result.trajectory
    return tutorial_common.render_replay_animation(
        participants,
        map_,
        frames,
        ego_id,
        plans=result.plans,
        resolution=resolution,
        fps=fps,
        title_prefix="LimSim takeover",
    )

### Example 1: WOMD - validation_interactive, Scenario 2 (the shared scene)

The junction all four behavior demos are measured on: LimSim, InterSim and SMART take over the same vehicle `8` at the same frame, so their predictions can be compared directly. The take-over errors printed above (ADE@2s 0.694 m) are the tightest of the four - LimSim is a receding-horizon planner tracking the lane, not a sampled future generator, and on a nearly straight approach it stays close to the recording.


In [5]:
ani_womd = run_takeover_scenario(
    WOMDParser(),
    scenario_id=tutorial_common.COMPARISON_SCENARIO,
    file_name=tutorial_common.COMPARISON_FILE,
    folder=f"../../../data/womd/uncompressed/{tutorial_common.COMPARISON_SPLIT}",
    ego_id=tutorial_common.COMPARISON_EGO,
    takeover_frame_ms=tutorial_common.COMPARISON_FRAME_MS,
)
ani_womd

Parsing scenario ...
  participants: 34,  frames: (0, 8976)
  takeover target: 8  (color: light-pink)
  MPC: 200 cycles  |  playback: 91 frames  (0-8976 ms, ~9s)
  take-over vs the recorded future:  ADE@2s 0.694 m  FDE@2s 1.967 m  (20 steps)
                                    ADE@5s 6.053 m  FDE@5s 18.184 m  (50 steps)


### Example 2: WOMD - validation_interactive, Scenario 9 (the yielding case)

A second scene from the same shard, picked for the opposite obligation: the ego opens almost stationary, and the relation it belongs to (`2608 -> 2478`) has it as the reactor rather than the one already committed. 42 participants and 311 lanes make it the busiest scene the demo replays.


In [6]:
ani_womd_9 = run_takeover_scenario(
    WOMDParser(),
    scenario_id=tutorial_common.SECOND_SCENARIO,
    file_name=tutorial_common.COMPARISON_FILE,
    folder=f"../../../data/womd/uncompressed/{tutorial_common.COMPARISON_SPLIT}",
    ego_id=tutorial_common.SECOND_EGO,
    takeover_frame_ms=tutorial_common.SECOND_FRAME_MS,
)
ani_womd_9

Parsing scenario ...
  participants: 42,  frames: (0, 9000)
  takeover target: 2478  (color: light-pink)
  MPC: 200 cycles  |  playback: 91 frames  (0-9000 ms, ~9s)
  take-over vs the recorded future:  ADE@2s 0.065 m  FDE@2s 0.118 m  (20 steps)
                                    ADE@5s 0.572 m  FDE@5s 3.578 m  (50 steps)


### Example 3: inD - Location 1, Recording 07 (off-domain, resampled)

A German urban junction at 25 Hz, so two things change at once: the domain is new and the rate is new - the runner resamples it onto the model's 100 ms lattice first. The take-over error is much larger here (ADE@2s 6.173 m) than on WOMD, which is what an off-domain scene looks like for a rule-based planner: the lane geometry and the traffic it was tuned on are not the ones it is being asked about.


In [7]:
ani_ind = run_takeover_scenario(
    LevelXParser("inD"),
    file_name=7,
    folder="../../data/inD/data",
    map_path="../../data/inD_map/inD_1.osm",
    map_config=IND_MAP_CONFIG["inD_1"],
    ego_id=12,
    fps=1000.0 / LIMSIM_CFG.step_ms,  # the playback frames are on the model's 100 ms lattice
)
ani_ind

Parsing scenario ...
  loading map from ../../data/inD_map/inD_1.osm
  participants: 212,  frames: (np.int64(0), np.int64(1055240))
  takeover target: 12  (color: light-pink)
  MPC: 200 cycles  |  playback: 64 frames  (10300-16600 ms, ~6s)
  take-over vs the recorded future:  ADE@2s 0.209 m  FDE@2s 0.454 m  (20 steps)
                                    ADE@5s 8.476 m  FDE@5s 30.015 m  (50 steps)


### Example 4: nuPlan - Boston Intersection (off-domain, resampled)

nuPlan sits at the other end of the map scale: one log carries a whole city's map, and the scenario is a window cut out of it. The window is the longest pass through an intersection in the log, recomputed from the log's `scenario_tag` table at run time rather than stored, because the parser stamps frames relative to `datetime(2021, 1, 1)` **in the local timezone** - a hard-coded window would mean a different stretch of road on another machine.


In [8]:
# nuPlan's map lives in its own .gpkg, not next to the log.
nuplan_window = tutorial_common.nuplan_intersection_window(
    f"../../data/nuplan/data/cache/{tutorial_common.NUPLAN_SCENARIO_FOLDER}"
    f"/{tutorial_common.NUPLAN_SCENARIO_FILE}"
)

ani_nuplan = run_takeover_scenario(
    NuPlanParser(),
    file_name=tutorial_common.NUPLAN_SCENARIO_FILE,
    folder=f"../../data/nuplan/data/cache/{tutorial_common.NUPLAN_SCENARIO_FOLDER}",
    map_file="map.gpkg",
    map_folder=f"../../data/nuplan/maps/{tutorial_common.NUPLAN_SCENARIO_MAP}",
    time_range=nuplan_window,
    ego_id=tutorial_common.NUPLAN_SCENARIO_EGO,
    num_seconds=10,
)
ani_nuplan

Parsing scenario ...


  participants: 100,  frames: (20572567849, 20572584199)
  takeover target: 26  (color: light-pink)
  MPC: 100 cycles  |  playback: 111 frames  (20572567849-20572578849 ms, ~11s)
  take-over vs the recorded future:  ADE@2s 0.740 m  FDE@2s 2.011 m  (20 steps)
                                    ADE@5s 3.010 m  FDE@5s 2.747 m  (50 steps)


## Quick Configurations

| Use Case | Key Settings |
|----------|-------------|
| **Fast preview** | `mcts_iterations=50`, `terminal_depth=3` - a shallow search, good enough to see the behaviour |
| **Demo defaults** | `mcts_iterations=200`, `terminal_depth=4`, `max_group_size=3`, `interaction_distance=30` |
| **Wider interaction** | `interaction_distance` and `max_group_size` grow the joint decision, and the search cost with it |
| **No Frenet refinement** | `use_frenet_refinement=False` keeps the raw planner output |
